# Análisis de atención — BETO + LoRA

Visualización de qué tokens recibe más atención el modelo final al
clasificar ejemplos del test set.

In [ ]:
!git clone https://github.com/camistrika/BETO_HUMOR.git
%cd BETO_HUMOR
!pip install -e . -q

In [ ]:
!pip install -q torchao --upgrade
!pip install -q transformers peft datasets scikit-learn pyyaml

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

from betohumor.config import DataConfig, BetoConfig
from betohumor.utils import set_seed
from betohumor.dataset import load_and_split

## 1. Cargar el modelo final LoRA

In [ ]:
LORA_PATH = 'results/lora/final_lora'

data_config = DataConfig()
beto_config = BetoConfig()
set_seed(data_config.seed)

df_train, df_val, df_test = load_and_split(data_config)
tokenizer = AutoTokenizer.from_pretrained(LORA_PATH)

base_model = AutoModelForSequenceClassification.from_pretrained(
    beto_config.base_model, num_labels=beto_config.num_labels,
)
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

## 2. Función para extraer atención de un texto

Como el modelo es `PeftModel` (LoRA encima de BETO), hay que acceder al
modelo base para poder pedir `output_attentions=True`.

In [ ]:
def get_attention(text, tokenizer, model, data_config, device):
    inputs = tokenizer(
        text,
        return_tensors='pt',
        max_length=data_config.max_length,
        truncation=True,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # PeftModel envuelve al modelo base; accedemos a base_model para
    # poder pedir output_attentions correctamente.
    inner_model = model.base_model.model if hasattr(model, 'base_model') else model

    with torch.no_grad():
        outputs = inner_model(**inputs, output_attentions=True)

    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    attentions = [att.squeeze(0).cpu() for att in outputs.attentions]  # lista de [num_heads, seq_len, seq_len], una por capa
    return tokens, attentions

## 3. Token importance desde [CLS]
Promedio de atención que cada token recibe desde [CLS], en la última capa,
promediando todas las cabezas.

In [ ]:
def get_cls_importance(attentions, layer=-1):
    attn_layer = attentions[layer]            # [num_heads, seq_len, seq_len]
    cls_attn = attn_layer[:, 0, :]             # atención desde el token [CLS] (posición 0)
    importance = cls_attn.mean(dim=0).numpy()  # promedio entre cabezas
    return importance

def plot_token_importance(tokens, importance, title=''):
    norm = importance / importance.max()
    order = np.argsort(norm)
    fig, ax = plt.subplots(figsize=(7, max(4, len(tokens) * 0.3)))
    ax.barh([tokens[i] for i in order], norm[order], color='#4C72B0')
    ax.set_xlabel('Importancia normalizada (atención desde [CLS])')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return fig

## 4. Seleccionar ejemplos representativos
Algunos de humor, algunos de no-humor.

In [ ]:
ejemplos_humor    = df_test[df_test[data_config.label_col] == 1].sample(3, random_state=data_config.seed)
ejemplos_no_humor = df_test[df_test[data_config.label_col] == 0].sample(3, random_state=data_config.seed)

ejemplos = pd.concat([ejemplos_humor, ejemplos_no_humor])
ejemplos[[data_config.text_col, data_config.label_col]]

## 5. Token importance por ejemplo

In [ ]:
os.makedirs('results/lora/attention', exist_ok=True)

for i, (_, row) in enumerate(ejemplos.iterrows()):
    text = row[data_config.text_col]
    label = 'Humor' if row[data_config.label_col] == 1 else 'No humor'

    tokens, attentions = get_attention(text, tokenizer, model, data_config, device)
    importance = get_cls_importance(attentions, layer=-1)

    print(f"\n[{label}] {text}")
    fig = plot_token_importance(tokens, importance, title=f'[{label}] {text[:50]}')
    fig.savefig(f'results/lora/attention/token_importance_{i}.png', dpi=150)

## 6. Heatmap de atención (capa final, una cabeza)
Para uno o dos ejemplos puntuales, ver la matriz completa token-a-token.

In [ ]:
def plot_attention_heatmap(tokens, attentions, layer=-1, head=0, title=''):
    attn = attentions[layer][head].numpy()  # [seq_len, seq_len]

    fig, ax = plt.subplots(figsize=(max(6, len(tokens)*0.5), max(5, len(tokens)*0.5)))
    im = ax.imshow(attn, cmap='viridis')
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=90, fontsize=8)
    ax.set_yticklabels(tokens, fontsize=8)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()
    return fig

In [ ]:
# Elegir un par de ejemplos representativos para el heatmap completo
for i, (_, row) in enumerate(ejemplos.head(2).iterrows()):
    text = row[data_config.text_col]
    label = 'Humor' if row[data_config.label_col] == 1 else 'No humor'

    tokens, attentions = get_attention(text, tokenizer, model, data_config, device)
    fig = plot_attention_heatmap(tokens, attentions, layer=-1, head=0, title=f'[{label}] {text[:40]} — capa final, cabeza 0')
    fig.savefig(f'results/lora/attention/heatmap_{i}.png', dpi=150)

## 7. Comparar con las palabras top del NBSVM
¿BETO presta atención a las mismas palabras léxicas (chiste, jajaja) o a
estructuras distintas (negaciones, ironía)? Completar con las palabras
top que viste en `baseline_nbsvm.ipynb`.

In [ ]:
# Completar con las palabras reales del NBSVM (sección de top palabras)
PALABRAS_TOP_NBSVM_HUMOR = ['chiste', 'jajaja', 'jaimito', 'pepito'] 

for _, row in ejemplos.iterrows():
    text = row[data_config.text_col]
    tokens, attentions = get_attention(text, tokenizer, model, data_config, device)
    importance = get_cls_importance(attentions, layer=-1)

    tokens_lower = [t.lower() for t in tokens]
    coincidencias = [t for t in tokens_lower if t in PALABRAS_TOP_NBSVM_HUMOR]
    if coincidencias:
        print(f'"{text[:60]}" contiene palabras top del NBSVM: {coincidencias}')
        for palabra in coincidencias:
            idx = tokens_lower.index(palabra)
            print(f'  Importancia de "{palabra}": {importance[idx]/importance.max():.3f} (normalizada)')

## 8. Descargar resultados

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('attention_results', 'zip', 'results/lora/attention')
files.download('attention_results.zip')

## 9. Observaciones cualitativas

_Completar después de revisar los resultados: ¿el modelo se fija en las
mismas palabras léxicas que el NBSVM, o en estructuras distintas
(negaciones, signos de puntuación, remates)? ¿Hay diferencia de patrón
entre ejemplos de humor y no-humor?_